# Deep Learning in Audio Applications
### Teach machines to listen: your own audio-to-text notebook

You will take speech, turn it into text, see where that breaks, build the two ideas that let a computer read a sequence
(an **RNN**, then an **LSTM**), and end with **meeting minutes** from a conversation with two speakers.

Run the cells from top to bottom. Cells marked **YOUR TURN** have one line for you to write; a check tells you straight away if it is right.

| section | content | agenda block(s) | minutes |
|---|---|---|---|
| 1 | Audio to text with Python | 2 (part) | ~6 |
| 2 | Where transcription struggles | 2 (part) | ~4 |
| 3 | RNNs: reading words as a sequence (+ Hands-on A, + why memory fades) | 3, 4, 5 | 15 + 10 + 7 |
| 4 | LSTMs: fixing the short memory (+ teaching it, + Hands-on B) | 6, 7, 8 | 20 + 10 + 20 |
| 5 | Meeting minutes (+ your own meeting) | 9 | 15 |


## 0. Setup (run once, about a minute)
On Colab: **Runtime > Change runtime type > T4 GPU** makes Whisper several times faster (the free CPU works too, just slower).
Upload `workshop_files.zip` from the facilitators to the Files panel on the left before running this cell.

In [ ]:
import os, sys, shutil, subprocess, importlib.util, zipfile
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")                  # avoids a library clash on some Windows installs

if importlib.util.find_spec("whisper") is None:                       # Whisper: speech-to-text from OpenAI
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai-whisper"], check=True)
if os.path.exists("workshop_files.zip") and not os.path.exists("prototype"):
    zipfile.ZipFile("workshop_files.zip").extractall(".")              # the code and audio for this workshop
PROTO = next((p for p in ("prototype", "../prototype") if os.path.isdir(p)), None)   # ../prototype when run from the repo
PROTO or sys.exit("Upload workshop_files.zip to the Files panel (left), then run this cell again.")
sys.path.insert(0, PROTO)
print("ffmpeg (reads audio files):", "found" if shutil.which("ffmpeg") else "NOT FOUND - install it (Colab has it already)")
print("ready")

## 1. Audio to text with Python
*Facilitator-run, during the talk -- sections 1-2 together are Agenda block 2, about 10 minutes*
*(roughly 6 min here, 4 min in section 2). Participants: run these cells too (about a minute on*
*a T4); the later sections reuse their results. Nothing in either section is YOUR TURN.*

Whisper is a ready-made speech-to-text model. Three lines: load it, give it a file, read the text.
Bigger models hear more but run slower: `tiny.en`, `base.en`, `small.en`, `medium.en` (the `.en` ones know only English). We pick a size that fits your machine.


In [ ]:
import torch, whisper

MODEL = "small.en" if torch.cuda.is_available() else "base.en"        # a GPU can afford the bigger model
print("using", MODEL, "on", "the GPU" if torch.cuda.is_available() else "the CPU")
model = whisper.load_model(MODEL)
result = model.transcribe("audio/sample_meeting.wav", language="en")
print(result["text"])

### What Whisper is reading
A spectrogram is like sheet music for speech: **time runs left to right, pitch runs bottom to top**.
Here is the waveform and the spectrogram for a short clip: three sentences from the meeting.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

wave_samples = whisper.load_audio("audio/sample_short.wav")
t = np.arange(len(wave_samples)) / 16000

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
ax1.plot(t, wave_samples, linewidth=0.5)
ax1.set_ylabel("amplitude")
ax1.set_title("Waveform")
with np.errstate(divide="ignore"):      # a silent stretch gives log10(0); matplotlib handles it, just don't warn
    ax2.specgram(wave_samples, Fs=16000, cmap="magma")
ax2.set_ylabel("pitch (Hz)")
ax2.set_xlabel("time (s)")
ax2.set_title("Spectrogram -- time left to right, pitch bottom to top")
plt.tight_layout()
plt.show()


**Order matters.** Play a short clip forwards, then backwards -- same sounds, but backwards it
stops being words. Whisper reads its spectrogram in this same left-to-right order; reverse it and
the "words" it would read make no sense either.


In [ ]:
from IPython.display import Audio, display

clip = wave_samples[:8000]          # about half a second -- roughly one word
print("forwards:")
display(Audio(clip, rate=16000))
print("backwards:")
display(Audio(clip[::-1], rate=16000))


**What Whisper-style models actually see.** Not the waveform directly -- a short window every
10 ms, turned into a handful of numbers (an MFCC frame, the classic speech-model feature; Whisper
itself uses a similar but larger spectrogram feature). This is the picture above, in the form a
model reads.


In [ ]:
from audio_features import mfcc

samples = whisper.load_audio("audio/sample_short.wav")
frames = mfcc(samples)
print(f"{len(samples) / 16000:.0f} seconds of audio  ->  {frames.shape[0]} frames x {frames.shape[1]} numbers (one frame every 10 ms)")


Our pipeline wants more than one block of text: it wants **who said what**. `get_transcript` listens for pauses and
starts a new turn each time one person stops and the next starts (here: a pause longer than 1.5 s).

In [ ]:
os.environ["WHISPER_MODEL"] = MODEL          # the pipeline's transcriber reads this
from transcriber import get_transcript

transcript = get_transcript("audio/sample_meeting.wav", pause_gap=1.5)
print(transcript)

## 2. Where transcription struggles
*Facilitator-run, about 4 of block 2's 10 minutes (see section 1's note).*

Real recordings are not clean. The three things that hurt most: **background noise**, **a poor microphone**, and **accents**.
We score by how many of the words Whisper gets right (`word_accuracy`: 1.0 = every word).


In [ ]:
import numpy as np
from audio_check import word_accuracy

SAID = ("Venue is confirmed for the 14th, catering still pending. "
        "Can you follow up with catering by Friday? Yes, I'll email them today.")
clean = whisper.load_audio("audio/sample_short.wav")          # the same three sentences, 14 seconds

def hear(samples):
    return model.transcribe(samples, language="en")["text"].strip()

print("clean      ", f"{word_accuracy(SAID, hear(clean)):.0%}", "|", hear(clean))

**Noise.** Add random hiss to the recording. How loud can it get before Whisper breaks? (try 0.02, 0.05, 0.1, 0.2)

In [ ]:
NOISE = 0.1
noisy = (clean + NOISE * np.random.default_rng(0).normal(size=clean.shape)).astype("float32")
heard = hear(noisy)
print(f"noise {NOISE}:", f"{word_accuracy(SAID, heard):.0%}", "|", heard)

**A poor microphone.** Keep only every 4th sample, like a cheap mic or a phone line. The high sounds (s, t, f) disappear.

In [ ]:
poor = np.repeat(clean[::4], 4).astype("float32")
heard = hear(poor)
print("poor mic  :", f"{word_accuracy(SAID, heard):.0%}", "|", heard)

**Accents.** Record yourself on your phone reading the same three sentences, with about 2 seconds between them:

> Venue is confirmed for the 14th, catering still pending. / Can you follow up with catering by Friday? / Yes, I'll email them today.

Upload it to the Files panel, put its name below, and compare with the clean voice. What did Whisper get wrong, and why?

In [ ]:
MY_VOICE = None            # for example "my_voice.m4a"
if MY_VOICE:
    mine = whisper.load_audio(MY_VOICE)
    heard = hear(mine)
    print("my voice  :", f"{word_accuracy(SAID, heard):.0%}", "|", heard)
else:
    print("Set MY_VOICE to a file you uploaded, then run this cell again.")

**Kahoot: which gives the most accurate transcript?** Four clips of the exact same sentence --
an ideal recording and three noisy versions. The facilitator plays each one and the room votes on
the ranking, best to worst, before the answer key cell below is run.


In [ ]:
NOISY_CLIPS = [
    ("1. ideal", "noisy_1_ideal.wav"),
    ("2. background hiss", "noisy_2_hiss.wav"),
    ("3. poor microphone", "noisy_3_poor_mic.wav"),
    ("4. a second voice underneath", "noisy_4_second_voice.wav"),
]

print("Vote now: which order, best to worst, before running the next cell?")
print()
for label, filename in NOISY_CLIPS:
    print(label)
    display(Audio(f"audio/{filename}"))


**Answer key.** Measured live, same as every other number in this notebook -- not assumed.
The percentages depend strongly on the Whisper model size -- `MODEL` in section 1 (`base.en`,
`small.en` or `medium.en`), not a fixed property of the clips. All three rows below are CPU runs
(no GPU available when this was built):

| model | ideal | second voice | hiss | poor mic |
|---|---|---|---|---|
| `base.en` | 100% | 61% | 52% | 35% |
| `small.en` (this notebook's GPU default) | 100% | 87% | 83% | 22% |
| `medium.en` | 100% | 91% | 83% | 35% |

This order -- ideal, then the second voice underneath, then hiss, then the poor microphone --
held on the three sizes we tried (CPU runs; on `small.en` the second voice and hiss are only
4 points apart, so check it on the day's Colab, with its GPU, before the vote). The exact
percentages are only for this machine's run; run the cell below to get yours.


In [ ]:
for label, filename in NOISY_CLIPS:
    heard = hear(whisper.load_audio(f"audio/{filename}"))
    print(f"{label:32s} {word_accuracy(SAID, heard):.0%}  | {heard}")


## 3. RNNs: reading words as a sequence (Agenda block 3, 15 min)
When you read a sentence, you don't reread every word from scratch -- you carry a running summary
in your head and update it with each new word. An RNN does the same: it reads one **word** at a
time and carries a small "summary" (its memory) forward:

`new memory = mix of (this word) + (old memory)`, written as `h = tanh(W_x·x + W_h·h_prev + b)`

(Whisper reads its spectrogram the same way, left to right -- the same "order matters" idea as the
reversed word in section 1, just for sound instead of words.)


In [ ]:
from lstm_model import model_words

sentence = "Can you follow up with catering by Friday?"
words = model_words(sentence)
vocab = {w: i for i, w in enumerate(sorted(set(words)))}
numbers = [vocab[w] for w in words]

print("words  :", words)
print("numbers:", numbers, " (each word -> its own number, via a vocabulary)")

# Step a tiny RNN over these words, one at a time, to show "reads one word at a time" --
# toy random weights and embeddings, not the trained model (that comes in section 4).
rng = np.random.default_rng(0)
dim = 4
toy_embeddings = rng.normal(size=(len(vocab), dim))
W_x_toy, W_h_toy, b_toy = rng.normal(scale=0.5, size=(dim, dim)), rng.normal(scale=0.5, size=(dim, dim)), np.zeros(dim)

print()
print("Stepping the RNN one word at a time:")
memory = np.zeros(dim)
for word, number in zip(words, numbers):
    x = toy_embeddings[number]
    memory = np.tanh(W_x_toy @ x + W_h_toy @ memory + b_toy)
    print(f"  after {word!r:10s} memory = {np.round(memory, 2)}")


### YOUR TURN (Part A): one step of an RNN (Agenda block 4, Hands-on A, 10 min)
At every step the RNN mixes **the new input** with **its old memory**:

`new memory = tanh( W_x · x  +  W_h · old memory  +  b )`

In numpy `·` is `@`, and tanh is `np.tanh`. Write that one line.

In [ ]:
import numpy as np, hands_on

def rnn_step(x, h_prev, W_x, W_h, b):
    # TODO: return the new memory (the line above, in numpy)
    return None

hands_on.rnn_step = rnn_step
hands_on.MEMORY_ON = True
hands_on.part_a()

Now **break the memory**: set `hands_on.MEMORY_ON = False` in the cell above (no old memory in the formula) and run it again. What happens to the first word?

### Why simple memory fades (Agenda block 5, 7 min)
Think of a game of broken telephone: pass a message through enough people and it's gone. An
RNN's memory fades the same way at every step. This is an **illustration** of that idea, not a
measured property of the trained model (which stacks 5 LSTM layers, not one plain RNN step) --
see the LSTM section below for what actually fixes it.

(Part A's toy RNN above kept its own share each step, around 58% on its random weights -- that
number is specific to that toy setup. The 0.49 below is what the untrained LSTM's own gates
measured, before training, on this project's actual model.)

This is exactly why **long-range dependencies** matter: if an early line in a meeting sets up
something a much later line depends on, a memory that has already faded can't connect them --
which is the problem the LSTM's gates (below) are built to fix.


In [ ]:
n = np.arange(48)
for r, note in [(0.49, "the model's starting (untrained) settings"), (0.95, "keeping 95% each step")]:
    retained = r ** n
    plt.plot(n, retained * 100, label=f"r = {r} ({note}): {retained[-1]:.1%} left after 47 steps")
plt.xlabel("steps (n)")
plt.ylabel("% of the original signal retained")
plt.title("Memory retention: r^n (illustration, not a measured property of the trained model)")
plt.legend()
plt.ylim(0, 100)
plt.show()

print(f"0.49^47 = {0.49**47:.4%}   0.95^47 = {0.95**47:.1%}")


## 4. LSTMs: fixing the short memory (Agenda block 6, 20 min)
An RNN's memory fades a little at every step, so words from long ago are lost. An LSTM keeps its
memory in a **notebook** (the cell state) with three switches at every step: the **eraser**
(forget gate, `f`) decides what to rub out, the **pencil** (input gate, `i`) decides what new
thing to write down, and the **voice** (output gate, `o`) decides what to say out loud right now.

`new notebook = (what you keep) × (old notebook) + (what you add) × (new note)`, written as
`c = f·c_prev + i·g`

Our model stacks 5 of these layers, like an hourglass: the middle layer covers more words per
step (folded together), giving memory fewer steps to survive there by design -- an architecture
idea, not something this trained classifier measures or proves on its own. A `<eot>` token
between turns is the "speaker changed" bookmark -- the model's only signal that someone new is
talking.


In [ ]:
from lstm_model import run_pipeline, load_trained_model, tokenize_transcript, pad_to_multiple_of_4

turns, output, turn_ends = run_pipeline(transcript=transcript)
trained = load_trained_model()
vocab = {w: i for i, w in enumerate(trained["vocab"])}
tokens, _ends = tokenize_transcript(turns)
x = trained["embeddings"][[vocab.get(w, 0) for w in tokens]]
_hourglass_out, hourglass_layers = trained["model"].forward(pad_to_multiple_of_4(x))

print("The hourglass: 5 layers, each with its own units and time steps (this notebook's own meeting):")
print(f"  {'layer':<9}{'units':>7}{'time steps':>13}")
for name, out in hourglass_layers.items():
    steps, units = out.shape
    print(f"  {name:<9}{units:>7}{steps:>13}")

layer_names = list(hourglass_layers.keys())
layer_units = [out.shape[1] for out in hourglass_layers.values()]
plt.figure(figsize=(6, 2.5))
plt.bar(layer_names, layer_units)
plt.ylabel("units")
plt.title("The hourglass shape: units per layer")
plt.show()


`<eot>` is the only signal the model gets that a new speaker is talking -- there is no speaker
name in the input (`tokenize_transcript` drops it; `prototype/demo.py` only puts the real names
back into the minutes afterwards). Here it is in a 2-turn exchange:


In [ ]:
from lstm_model import TURN_BOUNDARY_TOKEN

example = [("Alex", "Can you follow up with catering by Friday?"), ("Priya", "Yes, I'll email them today.")]
example_tokens, example_ends = tokenize_transcript(example)
print(example_tokens)
print(f"{TURN_BOUNDARY_TOKEN!r} marks \"speaker changed\" between the two turns "
      f"(turn_end_indices = {example_ends}).")


In [ ]:
from lstm_model import fold_time, unfold_time
x = np.arange(16.0).reshape(8, 2)
print("8 steps of 2 numbers, folded:", fold_time(x).shape, "(4 steps of 4); unfolded back is identical:", np.array_equal(unfold_time(fold_time(x)), x))

### Teaching it, and catching it out (Agenda block 7, 10 min)
Training works like marking practice papers: the model sees a meeting line with the right
label, guesses, is told how wrong it was, and adjusts a little -- repeated over many lines.

`label -> prediction -> loss -> update`

How wrong is measured as `loss = -log(p(the model gave the correct label))` (the natural log):
confident and right is a tiny loss, confident and wrong is a large one. Worked example, on this
notebook's own meeting:


In [ ]:
from lstm_model import label_probabilities
from meeting_minutes import label_turns_stub

gold = label_turns_stub(turns)
probs = label_probabilities(turns, trained)

for (speaker, text), g, p in zip(turns, gold, probs):
    if "email them today" in text or "Venue is confirmed" in text:
        correct = g["category"]
        p_correct = p[trained["labels"].index(correct)]
        print(text)
        print(f"  gold = {correct}, p(correct) = {p_correct:.4f}, loss = {-np.log(p_correct):.2f}")
        print()


The model is confident AND right on "Yes, I'll email them today." (tiny loss), and confident
AND **wrong** on "Venue is confirmed for the 14th..." (a decision stated as a plain fact, no
"agree" or "let's" in it) -- a large loss, exactly the kind of line that pulls training the most.

**Testing it on lines it has never seen.** A simple keyword checklist (does the line contain
"confirmed", "agreed", ...?) versus the trained model, on a held-out meeting -- a student
council, never used in training. This is the team's **best-case** held-out meeting for the
trained model, not a typical one: across all 17 held-out meetings (`holdout_data.py`) the
trained LSTM scores 131/149 (0.879) and 104/122 (0.852) (`python prototype/run_tests.py`
check k) against the keyword checklist's 111/149 (0.745) and 93/122 (0.762) (a per-meeting
count over `holdout_data.py`, not printed by any existing script) -- a real, solid improvement,
but nowhere near 17/18. On 3 of those 17 meetings the keyword checklist actually scores
*better* than the trained model.


In [ ]:
from pipeline import load_text
from meeting_minutes import label_turns_keywords, label_turns_model

held_out_text, held_out_gold = load_text(f"{PROTO}/demo_student_council.txt")
held_out_turns, held_out_output, held_out_ends = run_pipeline(transcript=held_out_text)
kw_labels = label_turns_keywords(held_out_turns)
lstm_labels = label_turns_model(held_out_turns, held_out_output, held_out_ends)

kw_right = sum(k["category"] == g for k, g in zip(kw_labels, held_out_gold))
lstm_right = sum(l["category"] == g for l, g in zip(lstm_labels, held_out_gold))
print(f"keyword rules: {kw_right}/{len(held_out_turns)} turns right")
print(f"trained LSTM:  {lstm_right}/{len(held_out_turns)} turns right")
print()

for (speaker, text), g, l in zip(held_out_turns, held_out_gold, lstm_labels):
    if l["category"] != g:
        print(f"LSTM miss: {text!r}  (gold {g}, got {l['category']})")


The labelling task mostly needs about one previous turn (a bare "Sure." after a request) --
not a long memory. **We don't claim the trained LSTM remembers further back than a plain RNN**
on this task; the long-range-dependency idea from the memory-fade graph above is illustrated by
the toy experiments (Hands-on A and B), not measured by this classifier.

And as seen above: a decision reported as a plain fact ("Venue is confirmed...", "No refreshments
this time...") is where the trained model is weakest -- interestingly, the simple keyword
checklist actually catches "Venue is confirmed" (it contains the word "confirmed"), while the
trained model, which should in principle do better, misses it. Neither approach is perfect; that
is the honest result, not a reason to trust one blindly over the other.


### YOUR TURN (Part B): the LSTM notebook update (Agenda block 8, Hands-on B, 20 min)
`new notebook = (how much to keep) × (old notebook) + (how much to add) × (the new note)`

`f` is the eraser, `i` is the pencil, `g` is the new note, `c_prev` is the old notebook. Write that one line.

In [ ]:
def lstm_cell_update(f, i, g, c_prev):
    # TODO: return the new notebook (the line above, in numpy)
    return None

hands_on.lstm_cell_update = lstm_cell_update
hands_on.FORGET_KNOB = 0          # try -6, -3, 0, 3, 6: negative erases more, positive keeps more
hands_on.part_b()

Turn the knob to **-6** and **+6** and run the cell again. What happens to the first word, and to the trained model's score? Why do both extremes hurt it?

**All three settings, compared.** Run the knob at -6, 0 and +6 in one go, and look at both
numbers each time: the toy gap at word 12, and the trained model's score on the test meetings.
"Keep a lot" (+6) is **not** "better memory" -- the trained model learned its own eraser
setting at 0, so pushing the knob either way makes its labels worse.


In [ ]:
for knob in (-6, 0, 6):
    hands_on.FORGET_KNOB = knob
    hands_on.part_b()
    print()
hands_on.FORGET_KNOB = 0   # leave the knob back at its trained setting


## 5. Meeting minutes (Agenda block 9, 15 min)
Now put it together: your two-speaker transcript goes through the trained LSTM, which labels every turn as a
**decision**, an **action item** or plain **discussion**, and the minutes are written from the labels.

In [ ]:
from lstm_model import run_pipeline
from meeting_minutes import label_turns_model, generate_minutes

turns, output, turn_ends = run_pipeline(transcript=transcript)
labels = label_turns_model(turns, output, turn_ends)
for (speaker, text), label in zip(turns, labels):
    print(f"{label['category']:<12} {speaker}: {text}")
print()
print(generate_minutes(turns, labels))

**Look closely.** Did it find both decisions? (It misses decisions stated as plain facts, such as "Venue is confirmed", and
a bare "Sounds good." after a proposal. That is where a model trained on few examples is weakest.)


### Your own meeting (Agenda block 9, part)
Write 5 lines from a meeting of your own -- a CCA, project or club -- one turn per line,
`Name: what they said`. Labelling is optional, but if you add one, label **every** line with
`act` (action item), `dec` (decision) or `dis` (discussion) at the start -- the model was tested
on 17 held-out meetings (clubs, projects, group work) and gets about 0.85-0.88 of turns right,
but its vocabulary is small, so unusual words for your topic are read as `<unk>` and ignored.
Names show up as unseen words too, for the same reason -- the model never actually reads them
(they are dropped before it sees the text), so this is expected and harmless.


In [ ]:
import os
import tempfile
from lstm_model import load_trained_model
from pipeline import load_text, unknown_words

MY_MEETING = """
dis  Sam: Let's go over what's left before the trip.
dec  Lee: The venue is confirmed for Friday.
dis  Sam: Should we also book the shuttle?
act  Sam: Lee, can you call the shuttle company?
act  Lee: Sure.
"""

with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False, encoding="utf-8") as f:
    f.write(MY_MEETING)
    my_meeting_path = f.name
try:
    my_text, my_gold = load_text(my_meeting_path)
except ValueError as e:
    # Strip the temp-file path off the front of the message -- a participant doesn't need to
    # see where this notebook happened to save their typed text, just what to fix.
    message = str(e).replace(my_meeting_path, "").strip(" ,:")
    print("Couldn't read that meeting:", message)
    print("Hint: label every line or none of them, and write each line as 'Name: text'")
    print("(something after the colon).")
    my_text = None
finally:
    os.remove(my_meeting_path)

if my_text is not None:
    my_turns, my_output, my_ends = run_pipeline(transcript=my_text)
    my_labels = label_turns_model(my_turns, my_output, my_ends)

    for i, ((speaker, text), label) in enumerate(zip(my_turns, my_labels)):
        model_cat = label["category"]
        if my_gold:
            your_cat = my_gold[i]
            mark = "" if your_cat == model_cat else "  <- X"
            print(f"{your_cat:<12} (you)   {model_cat:<12} (model)   {speaker}: {text}{mark}")
        else:
            print(f"{model_cat:<12} (model)   {speaker}: {text}")
    if not my_gold:
        print()
        print("No labels given -- add act/dec/dis at the start of every line above and run again")
        print("to see where you and the model agree.")

    print()
    print(generate_minutes(my_turns, my_labels))

    unknown = unknown_words(my_turns)
    print()
    if unknown:
        print(f"Words the model never saw in training (read as <unk>): {', '.join(unknown)}")
    vocab_size = len(load_trained_model()["vocab"])
    print(f"The model knows {vocab_size} words; anything else is read as <unk>.")

    print()
    print("Pick one line it got wrong. Which EARLIER line should it have remembered?")


### Going further
* Change `MODEL` in section 1 to `small.en` or `medium.en`. How much does the transcript improve, and how much slower is it?
* Record a real two-person conversation (pause between speakers) and run it through `get_transcript`.
* Section 2 again with your own noise level or a different recording.